In [9]:
import pandas as pd

df = pd.read_csv("C:/Users/polar/OneDrive/Desktop/credit_risk_dataset.csv")

print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

df = df.drop_duplicates()

df["person_emp_length"] = df["person_emp_length"].fillna(
    df["person_emp_length"].median()
)

df["loan_int_rate"] = df["loan_int_rate"].fillna(
    df["loan_int_rate"].median()
)

df = df[df["person_age"].between(18, 100)]

df = df[df["person_emp_length"] >= 0]

df = df[df["person_emp_length"] <= df["person_age"]]

df = df[df["person_emp_length"] <= 40]

df = df[df["person_income"] > 0]
df = df[df["loan_amnt"] > 0]

df = df[df["loan_percent_income"] <= 1]

# -------------------------------
# Create new features
# -------------------------------

# Age groups
df["age_group"] = pd.cut(
    df["person_age"],
    bins=[18, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55+"],
    include_lowest=True
)

#quartiles
df["income_band"] = pd.qcut(
    df["person_income"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

# Risk flag based on loan-to-income ratio
df["high_risk"] = df["loan_percent_income"].apply(
    lambda x: "High Risk" if x > 0.35 else "Low Risk"
)


print("\nCleaned Dataset Shape:", df.shape)

print("\nSummary Statistics:")
print(df.describe())

print("\nAge Group Distribution:")
print(df["age_group"].value_counts())

print("\nIncome Band Distribution:")
print(df["income_band"].value_counts())

print("\nRisk Flag Distribution:")
print(df["high_risk"].value_counts())


df.to_csv("credit_risk_cleaned.csv", index=False)

print("\nCleaning completed successfully!")
print("Cleaned file saved as: credit_risk_cleaned.csv")
print("\nPreview:")
print(df.head())

Dataset Shape: (32581, 12)

Missing Values:
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Cleaned Dataset Shape: (32408, 15)

Summary Statistics:
         person_age  person_income  person_emp_length     loan_amnt  \
count  32408.000000   3.240800e+04       32408.000000  32408.000000   
mean      27.729203   6.589483e+04           4.760306   9592.690077   
std        6.204260   5.251859e+04           3.978729   6320.876563   
min       20.000000   4.000000e+03           0.000000    500.000000   
25%       23.000000   3.850000e+04           2.000000   5000.000000   
50%       26.000000   5.500000e+04     

In [18]:
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[column] = df[column].clip(lower=lower, upper=upper)

    return df

for col in ["person_income", "loan_amnt"]:
    df = cap_outliers_iqr(df, col)

In [20]:
print(df[["person_income", "loan_amnt"]].describe())

       person_income     loan_amnt
count   32408.000000  32408.000000
mean    62427.775827   9417.600592
std     31801.204453   5827.539472
min      4000.000000    500.000000
25%     38500.000000   5000.000000
50%     55000.000000   8000.000000
75%     79200.000000  12250.000000
max    140250.000000  23125.000000


In [22]:
df.to_csv("credit_risk_cleaned.csv", index=False)

print("Final cleaned dataset saved successfully!")

Final cleaned dataset saved successfully!


In [24]:
df.to_csv(
    r"C:/Users/polar/OneDrive/Desktop/credit_risk_cleaned.csv",
    index=False
)